# 🌿 Plant Disease Classification
## Beginner-Friendly Guide to Image Classification

**Project Goal:** Classify leaf images as healthy or diseased using deep learning

**Models We'll Use:**
- Custom CNN (Convolutional Neural Network)
- MobileNet (Lightweight pretrained model)
- EfficientNet (State-of-the-art pretrained model)

**Dataset:** PlantVillage - 38 classes of plant diseases

## Step 1: Install Required Libraries

In [ ]:
# Install necessary packages
!pip install tensorflow tensorflow-datasets matplotlib seaborn scikit-learn

print("✅ All libraries installed successfully!")

## Step 2: Import Libraries

In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

# Check TensorFlow version and GPU availability
print(f"TensorFlow Version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

## Step 3: Load PlantVillage Dataset

The PlantVillage dataset contains 54,306 images of healthy and diseased plant leaves across 38 categories.

In [ ]:
# Load the dataset (this will download it automatically)
print("Downloading PlantVillage dataset...")
(train_ds, val_ds, test_ds), dataset_info = tfds.load(
    'plant_village',
    split=['train[:70%]', 'train[70%:85%]', 'train[85%:]'],
    with_info=True,
    as_supervised=True
)

# Get dataset information
num_classes = dataset_info.features['label'].num_classes
class_names = dataset_info.features['label'].names

print(f"\n✅ Dataset loaded successfully!")
print(f"Number of classes: {num_classes}")
print(f"\nSample class names:")
for i, name in enumerate(class_names[:5]):
    print(f"  {i}: {name}")

## Step 4: Visualize Sample Images

In [ ]:
# Visualize sample images from the dataset
plt.figure(figsize=(15, 8))
for images, labels in train_ds.take(1):
    for i in range(12):
        plt.subplot(3, 4, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[labels[i]], fontsize=9)
        plt.axis('off')
plt.tight_layout()
plt.suptitle('Sample Plant Images from Dataset', fontsize=16, y=1.02)
plt.show()

## Step 5: Data Preprocessing

We'll resize images to 224x224 and normalize pixel values to [0, 1] range.

In [ ]:
# Configuration
IMG_SIZE = 224
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

# Preprocessing function
def preprocess_image(image, label):
    """Resize and normalize images"""
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    image = image / 255.0  # Normalize to [0, 1]
    return image, label

# Data augmentation for training (helps prevent overfitting)
def augment_image(image, label):
    """Apply random transformations to training images"""
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_flip_up_down(image)
    image = tf.image.random_brightness(image, 0.2)
    image = tf.image.random_contrast(image, 0.8, 1.2)
    return image, label

# Prepare datasets
train_ds = (
    train_ds
    .map(preprocess_image, num_parallel_calls=AUTOTUNE)
    .map(augment_image, num_parallel_calls=AUTOTUNE)
    .shuffle(1000)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

val_ds = (
    val_ds
    .map(preprocess_image, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

test_ds = (
    test_ds
    .map(preprocess_image, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

print("✅ Data preprocessing complete!")
print(f"Training batches: {tf.data.experimental.cardinality(train_ds).numpy()}")
print(f"Validation batches: {tf.data.experimental.cardinality(val_ds).numpy()}")
print(f"Test batches: {tf.data.experimental.cardinality(test_ds).numpy()}")

## Step 6: Model 1 - Custom CNN

A simple Convolutional Neural Network built from scratch.

In [ ]:
def create_custom_cnn():
    """Build a custom CNN model"""
    model = tf.keras.Sequential([
        # Input layer
        tf.keras.layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
        
        # Convolutional Block 1
        tf.keras.layers.Conv2D(32, (3, 3), activation='relu'),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.BatchNormalization(),
        
        # Convolutional Block 2
        tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.BatchNormalization(),
        
        # Convolutional Block 3
        tf.keras.layers.Conv2D(128, (3, 3), activation='relu'),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.BatchNormalization(),
        
        # Convolutional Block 4
        tf.keras.layers.Conv2D(256, (3, 3), activation='relu'),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.BatchNormalization(),
        
        # Flatten and Dense layers
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(512, activation='relu'),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(256, activation='relu'),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(num_classes, activation='softmax')
    ])
    
    return model

# Create and compile the model
cnn_model = create_custom_cnn()
cnn_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("Custom CNN Model Architecture:")
cnn_model.summary()

## Step 7: Model 2 - MobileNet (Transfer Learning)

MobileNet is a lightweight pretrained model, perfect for beginners and mobile deployment.

In [ ]:
def create_mobilenet_model():
    """Build MobileNet model with transfer learning"""
    # Load pretrained MobileNetV2 (without top classification layer)
    base_model = tf.keras.applications.MobileNetV2(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_top=False,
        weights='imagenet'
    )
    
    # Freeze the base model (we'll use pretrained features)
    base_model.trainable = False
    
    # Add custom classification head
    model = tf.keras.Sequential([
        base_model,
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dense(256, activation='relu'),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(num_classes, activation='softmax')
    ])
    
    return model

# Create and compile the model
mobilenet_model = create_mobilenet_model()
mobilenet_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("MobileNet Model Architecture:")
mobilenet_model.summary()

## Step 8: Model 3 - EfficientNet (State-of-the-art)

EfficientNet is a highly efficient and accurate model family.

In [ ]:
def create_efficientnet_model():
    """Build EfficientNet model with transfer learning"""
    # Load pretrained EfficientNetB0
    base_model = tf.keras.applications.EfficientNetB0(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_top=False,
        weights='imagenet'
    )
    
    # Freeze the base model
    base_model.trainable = False
    
    # Add custom classification head
    model = tf.keras.Sequential([
        base_model,
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dense(512, activation='relu'),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(num_classes, activation='softmax')
    ])
    
    return model

# Create and compile the model
efficientnet_model = create_efficientnet_model()
efficientnet_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("EfficientNet Model Architecture:")
efficientnet_model.summary()

## Step 9: Training Configuration

Set up callbacks for better training (early stopping, model checkpointing, etc.)

In [ ]:
# Training parameters
EPOCHS = 20

# Callbacks
def get_callbacks(model_name):
    """Create training callbacks"""
    return [
        # Stop training if validation accuracy doesn't improve
        tf.keras.callbacks.EarlyStopping(
            monitor='val_accuracy',
            patience=5,
            restore_best_weights=True,
            verbose=1
        ),
        # Reduce learning rate when stuck
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=3,
            verbose=1,
            min_lr=1e-7
        )
    ]

print("✅ Training configuration ready!")
print(f"Epochs: {EPOCHS}")
print(f"Batch size: {BATCH_SIZE}")

## Step 10: Train Model 1 - Custom CNN

In [ ]:
print("🚀 Training Custom CNN Model...\n")

history_cnn = cnn_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=get_callbacks('cnn'),
    verbose=1
)

print("\n✅ Custom CNN training complete!")

## Step 11: Train Model 2 - MobileNet

In [ ]:
print("🚀 Training MobileNet Model...\n")

history_mobilenet = mobilenet_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=get_callbacks('mobilenet'),
    verbose=1
)

print("\n✅ MobileNet training complete!")

## Step 12: Train Model 3 - EfficientNet

In [ ]:
print("🚀 Training EfficientNet Model...\n")

history_efficientnet = efficientnet_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=get_callbacks('efficientnet'),
    verbose=1
)

print("\n✅ EfficientNet training complete!")

## Step 13: Compare Training History

In [ ]:
# Plot training history comparison
def plot_training_history(histories, model_names):
    """Compare training histories of multiple models"""
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # Plot accuracy
    for history, name in zip(histories, model_names):
        axes[0].plot(history.history['accuracy'], label=f'{name} Train', linewidth=2)
        axes[0].plot(history.history['val_accuracy'], label=f'{name} Val', linestyle='--', linewidth=2)
    axes[0].set_title('Model Accuracy Comparison', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Plot loss
    for history, name in zip(histories, model_names):
        axes[1].plot(history.history['loss'], label=f'{name} Train', linewidth=2)
        axes[1].plot(history.history['val_loss'], label=f'{name} Val', linestyle='--', linewidth=2)
    axes[1].set_title('Model Loss Comparison', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# Compare all three models
plot_training_history(
    [history_cnn, history_mobilenet, history_efficientnet],
    ['Custom CNN', 'MobileNet', 'EfficientNet']
)

## Step 14: Evaluate Models on Test Set

In [ ]:
def evaluate_model(model, model_name, test_dataset):
    """Evaluate model performance on test set"""
    print(f"\n{'='*50}")
    print(f"Evaluating {model_name}")
    print('='*50)
    
    # Get predictions
    test_loss, test_accuracy = model.evaluate(test_dataset, verbose=0)
    
    print(f"\nTest Accuracy: {test_accuracy*100:.2f}%")
    print(f"Test Loss: {test_loss:.4f}")
    
    return test_accuracy, test_loss

# Evaluate all models
results = {}
results['Custom CNN'] = evaluate_model(cnn_model, 'Custom CNN', test_ds)
results['MobileNet'] = evaluate_model(mobilenet_model, 'MobileNet', test_ds)
results['EfficientNet'] = evaluate_model(efficientnet_model, 'EfficientNet', test_ds)

# Summary comparison
print("\n" + "="*50)
print("FINAL COMPARISON")
print("="*50)
for model_name, (acc, loss) in results.items():
    print(f"{model_name:15s} - Accuracy: {acc*100:6.2f}% | Loss: {loss:.4f}")

## Step 15: Visualize Results with Bar Chart

In [ ]:
# Create comparison bar chart
model_names = list(results.keys())
accuracies = [acc * 100 for acc, _ in results.values()]

plt.figure(figsize=(10, 6))
bars = plt.bar(model_names, accuracies, color=['#3498db', '#e74c3c', '#2ecc71'], alpha=0.8)
plt.ylabel('Test Accuracy (%)', fontsize=12, fontweight='bold')
plt.title('Model Performance Comparison', fontsize=14, fontweight='bold')
plt.ylim([0, 100])
plt.grid(axis='y', alpha=0.3)

# Add value labels on bars
for bar, acc in zip(bars, accuracies):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{acc:.2f}%',
             ha='center', va='bottom', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.show()

## Step 16: Make Predictions on Sample Images

In [ ]:
# Get a batch of test images
for images, labels in test_ds.take(1):
    sample_images = images[:9]
    sample_labels = labels[:9]
    break

# Make predictions with the best model (usually EfficientNet)
predictions = efficientnet_model.predict(sample_images)
predicted_labels = np.argmax(predictions, axis=1)

# Visualize predictions
plt.figure(figsize=(15, 10))
for i in range(9):
    plt.subplot(3, 3, i + 1)
    plt.imshow(sample_images[i])
    
    true_label = class_names[sample_labels[i]]
    pred_label = class_names[predicted_labels[i]]
    confidence = predictions[i][predicted_labels[i]] * 100
    
    color = 'green' if predicted_labels[i] == sample_labels[i] else 'red'
    
    plt.title(f'True: {true_label[:20]}...\nPred: {pred_label[:20]}...\nConf: {confidence:.1f}%',
              fontsize=9, color=color, fontweight='bold')
    plt.axis('off')

plt.suptitle('Sample Predictions (Green=Correct, Red=Incorrect)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Step 17: Confusion Matrix for Best Model

In [ ]:
# Get all predictions and true labels
all_predictions = []
all_labels = []

for images, labels in test_ds:
    preds = efficientnet_model.predict(images, verbose=0)
    all_predictions.extend(np.argmax(preds, axis=1))
    all_labels.extend(labels.numpy())

# Create confusion matrix
cm = confusion_matrix(all_labels, all_predictions)

# Plot confusion matrix (simplified for 38 classes)
plt.figure(figsize=(16, 14))
sns.heatmap(cm, annot=False, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names,
            cbar_kws={'label': 'Count'})
plt.title('Confusion Matrix - EfficientNet Model', fontsize=16, fontweight='bold', pad=20)
plt.ylabel('True Label', fontsize=12, fontweight='bold')
plt.xlabel('Predicted Label', fontsize=12, fontweight='bold')
plt.xticks(rotation=90, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.show()

print("\n✅ Confusion matrix created!")
print("Diagonal elements show correct predictions, off-diagonal show misclassifications.")

## Step 18: Classification Report

In [ ]:
# Generate detailed classification report
report = classification_report(all_labels, all_predictions, 
                               target_names=class_names, 
                               digits=3)

print("\n" + "="*80)
print("CLASSIFICATION REPORT - EfficientNet Model")
print("="*80)
print(report)

## Step 19: Save the Best Model

In [ ]:
# Save the EfficientNet model (usually the best performer)
model_save_path = '/content/best_plant_disease_model.h5'
efficientnet_model.save(model_save_path)

print(f"✅ Model saved to: {model_save_path}")
print("\nYou can download this model and use it later!")
print("To load: model = tf.keras.models.load_model('best_plant_disease_model.h5')")

## 🎉 Congratulations!

You've successfully completed a plant disease classification project!

### What You Learned:
1. ✅ Loading and preprocessing image datasets
2. ✅ Data augmentation techniques
3. ✅ Building custom CNN architectures
4. ✅ Transfer learning with pretrained models (MobileNet, EfficientNet)
5. ✅ Model training with callbacks
6. ✅ Model evaluation and comparison
7. ✅ Visualization of results

### Next Steps:
- Try fine-tuning the pretrained models (unfreeze some layers)
- Experiment with different augmentation techniques
- Test with your own plant images
- Deploy the model as a web app or mobile app
- Try other datasets from TensorFlow Datasets

### Resources:
- [TensorFlow Documentation](https://www.tensorflow.org/)
- [Keras API Reference](https://keras.io/api/)
- [PlantVillage Dataset](https://www.tensorflow.org/datasets/catalog/plant_village)

## Optional: Test on Your Own Images

Upload your own leaf images and test the model!

In [ ]:
from google.colab import files
from PIL import Image
import io

def predict_custom_image():
    """Upload and predict on custom images"""
    print("Please upload a leaf image...")
    uploaded = files.upload()
    
    for filename in uploaded.keys():
        # Load and preprocess image
        img = Image.open(io.BytesIO(uploaded[filename]))
        img = img.resize((IMG_SIZE, IMG_SIZE))
        img_array = np.array(img) / 255.0
        img_array = np.expand_dims(img_array, axis=0)
        
        # Make prediction
        prediction = efficientnet_model.predict(img_array, verbose=0)
        predicted_class = np.argmax(prediction)
        confidence = prediction[0][predicted_class] * 100
        
        # Display result
        plt.figure(figsize=(8, 6))
        plt.imshow(img)
        plt.title(f'Prediction: {class_names[predicted_class]}\nConfidence: {confidence:.2f}%',
                 fontsize=12, fontweight='bold')
        plt.axis('off')
        plt.show()
        
        # Show top 5 predictions
        top_5_idx = np.argsort(prediction[0])[-5:][::-1]
        print("\nTop 5 Predictions:")
        for i, idx in enumerate(top_5_idx, 1):
            print(f"{i}. {class_names[idx]}: {prediction[0][idx]*100:.2f}%")

# Uncomment to test with your own images
# predict_custom_image()